# Probability and Statistics from Scratch

**Mathematics for Machine Learning from First Principles · Lesson 3**

Data are noisy and predictions are uncertain. Statistics summarises what data show, probability describes uncertainty, and maximum likelihood estimation joins the two by choosing the parameters under which the observed data are most probable. That last idea is where the most common training losses come from: cross-entropy is a negative Bernoulli log-likelihood, and mean squared error corresponds to a Gaussian log-likelihood with a fixed variance. This notebook builds each piece from the standard library and checks it against Python's `statistics` module.

## Learning objectives

By the end of this notebook you will be able to:

1. compute the mean, median, variance and standard deviation, and say when to divide by $n$ and when by $n - 1$;
2. implement the Bernoulli and binomial probability mass functions and the normal density, and explain why a density can exceed 1;
3. draw reproducible samples with `random.Random(seed)` and watch sample averages settle down as the sample grows;
4. compute conditional probabilities and apply Bayes' theorem to a screening test;
5. compute covariance and correlation, and recognise what correlation cannot tell you;
6. derive and implement maximum likelihood estimates for Bernoulli and Gaussian data, and explain why the Gaussian variance estimate divides by $n$.

## Dependencies

**Tier 0 — Python standard library only.** The core cells import only `math`, `random`, `statistics`, `fractions`, `itertools`, `copy` and `numbers`. The `statistics` module is part of the standard library, so it serves as the reference that your own functions are checked against. Two optional cells compare the results with NumPy and plot them with Matplotlib. Each is labelled *Optional*, tagged `comparison` or `visualization`, checks that its library is installed before using it, and is never used by a later cell.

The notebook runs in JupyterLite (Python 3.14 inside your browser) and on your JupyterHub server (Python 3.13). It needs no internet access and runs from top to bottom in a few seconds. Every random number comes from a generator with a fixed seed, so you get the same results each time.

### How to use this notebook

Run the cells from top to bottom. Each implementation cell is followed by a test cell of plain `assert` statements. The exercises use the same `check` helper as the earlier lessons, defined in the next cell: it prints `not implemented yet` until you write an exercise, prints a pass line when your function is right, and raises `AssertionError` naming the failing call when it is wrong. Reference solutions are at the end.

In [ ]:
import random
import statistics

import copy
import math
import numbers


def _matches(result, expected, rel_tol, abs_tol):
    """True when result equals expected: exactly for integers and booleans,
    within a tolerance for floats, and entry by entry for lists and tuples."""
    if expected is None or isinstance(expected, bool):
        return result is expected
    if isinstance(expected, numbers.Real):
        if isinstance(result, bool) or not isinstance(result, numbers.Real):
            return False
        if isinstance(expected, int):
            return result == expected
        return math.isclose(result, expected, rel_tol=rel_tol, abs_tol=abs_tol)
    if isinstance(expected, (list, tuple)):
        return (
            isinstance(result, (list, tuple))
            and len(result) == len(expected)
            and all(_matches(r, e, rel_tol, abs_tol) for r, e in zip(result, expected))
        )
    return result == expected


def check(name, function, cases, *, rel_tol=1e-9, abs_tol=1e-9, strict=False):
    """Test an exercise function against a list of (arguments, expected) cases.

    arguments is a tuple of positional arguments, for example ([1, 2, 3],) for a
    function that takes one list. expected is the value the function should
    return, or an exception class such as ValueError that it should raise.
    Integers must match exactly; floats are compared with math.isclose.

    Prints "<name>: not implemented yet" while the function still raises
    NotImplementedError, prints a pass line when every case is right, and raises
    AssertionError with the failing call when a case is wrong. strict=True
    (used in the Solutions section) treats an unimplemented function as a failure.
    """
    for arguments, expected in cases:
        call = f"{name}({', '.join(repr(argument) for argument in arguments)})"
        wants_error = isinstance(expected, type) and issubclass(expected, Exception)
        # Work on copies of list and dict arguments so that a function which
        # changes its input is caught instead of silently corrupting later cases.
        supplied = tuple(
            copy.deepcopy(argument) if isinstance(argument, (list, dict)) else argument
            for argument in arguments
        )
        try:
            result = function(*supplied)
        except NotImplementedError:
            if strict:
                raise AssertionError(f"{call} raised NotImplementedError; finish the function first")
            print(f"{name}: not implemented yet")
            return
        except Exception as error:
            if wants_error and isinstance(error, expected):
                continue
            wanted = f"it to raise {expected.__name__}" if wants_error else repr(expected)
            raise AssertionError(
                f"{call} raised {type(error).__name__}: {error}; expected {wanted}"
            ) from error
        if wants_error:
            raise AssertionError(f"{call} returned {result!r}; expected it to raise {expected.__name__}")
        for before, after in zip(arguments, supplied):
            if isinstance(before, (list, dict)) and before != after:
                raise AssertionError(
                    f"{call} changed its input {before!r} into {after!r}; "
                    "return a new list instead of modifying the argument"
                )
        if not _matches(result, expected, rel_tol, abs_tol):
            raise AssertionError(f"{call} returned {result!r}; expected {expected!r}")
    summary = "the test" if len(cases) == 1 else f"all {len(cases)} tests"
    print(f"{name}: {summary} passed")



def raises_value_error(function, *arguments):
    """True when function(*arguments) raises ValueError (used by the tests below)."""
    try:
        function(*arguments)
    except ValueError:
        return True
    return False

## Theory

### Describing data

For values $x_1, \ldots, x_n$, the **mean** is $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$. The **median** is the middle value once the data are sorted, or the average of the two middle values when $n$ is even. One extreme value can drag the mean a long way, but it barely moves the median.

The **variance** measures spread as an average squared deviation from the mean, and there are two versions:

$$\sigma^2_{\text{pop}} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2, \qquad s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2.$$

Use the **population variance** $\sigma^2_{\text{pop}}$ when the data are the whole population you care about, or when you want the maximum likelihood estimate (see below). Use the **sample variance** $s^2$ to estimate the variance of a larger population from a sample. The deviations are measured from $\bar{x}$, which was fitted to the same data, so they are on average slightly smaller than deviations from the true mean $\mu$. In fact $\mathbb{E}\left[\sum_i (x_i - \bar{x})^2\right] = (n - 1)\,\sigma^2$, so dividing by $n - 1$ (Bessel's correction) makes $s^2$ **unbiased**: $\mathbb{E}[s^2] = \sigma^2$. The **standard deviation** is the square root of a variance, in the same units as the data.

The `statistics` module provides both versions: `pvariance` and `pstdev` divide by $n$, while `variance` and `stdev` divide by $n - 1$.

### Probability distributions

A **probability mass function** (pmf) gives the probability of each value of a discrete random variable.

* A **Bernoulli** variable is $1$ with probability $p$ and $0$ otherwise: $P(X = x) = p^x (1 - p)^{1 - x}$ for $x \in \{0, 1\}$. Its mean is $p$ and its variance is $p(1 - p)$.
* A **binomial** variable counts the successes in $n$ independent Bernoulli trials:

$$P(K = k) = \binom{n}{k}\, p^k (1 - p)^{n - k}, \qquad k = 0, 1, \ldots, n,$$

with mean $np$ and variance $np(1 - p)$. The coefficient $\binom{n}{k}$ counts the orders in which $k$ successes can occur, and `math.comb(n, k)` computes it.

A continuous variable has a **probability density function** (pdf) instead. The **normal** (Gaussian) density with mean $\mu$ and standard deviation $\sigma$ is

$$f(x \mid \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right).$$

A density is not a probability. Probabilities are areas under it, $P(a \le X \le b) = \int_a^b f(x)\,dx$, and the total area is $1$. A narrow density must therefore be tall, so $f$ can exceed $1$ when $\sigma$ is small.

### Sampling and the law of large numbers

A pseudo-random number generator produces a long, deterministic sequence of numbers that behaves like a random one. The **seed** chooses where the sequence starts, so the same seed gives the same "random" numbers every time. `random.Random(seed)` creates a generator object of your own. Pass that object into functions as an argument, rather than relying on the shared generator behind `random.random()`: the caller then controls the seed, and no other code can disturb the sequence. Python guarantees that `Random.random()` reproduces the same sequence for the same seed across versions, but other methods, such as `gauss` and `choice`, may change between releases. Tests of random code should therefore check statistical properties with a tolerance, or compare two runs that share a seed.

The **law of large numbers** says that the average of $n$ independent draws approaches the expected value as $n$ grows. For a Bernoulli($p$) variable, the typical error of the sample mean is the **standard error** $\sqrt{p(1 - p)/n}$, so a sample 100 times larger is only about 10 times more accurate.

### Conditional probability and Bayes' theorem

The probability of $A$ **given** that $B$ has happened is

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \qquad P(B) > 0.$$

When all outcomes are equally likely, this is a ratio of counts: $|A \cap B| / |B|$. In general $P(A \mid B) \ne P(B \mid A)$. **Bayes' theorem** turns one into the other. For a hypothesis $H$ and evidence $E$,

$$P(H \mid E) = \frac{P(E \mid H)\, P(H)}{P(E)}, \qquad P(E) = P(E \mid H)\, P(H) + P(E \mid \neg H)\, P(\neg H).$$

$P(H)$ is the **prior**, $P(E \mid H)$ the **likelihood**, $P(E)$ the **evidence** and $P(H \mid E)$ the **posterior**. When $H$ is rare, even an accurate test produces many false positives, because there are so many more people without the condition than with it.

### Covariance and correlation

The sample **covariance** of paired values $(x_i, y_i)$ measures whether they move together:

$$\operatorname{cov}(x, y) = \frac{1}{n - 1}\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y}).$$

It is positive when large $x$ tends to come with large $y$, and its units are those of $x$ times those of $y$. The **Pearson correlation** divides by both standard deviations to remove the units:

$$r = \frac{\operatorname{cov}(x, y)}{s_x\, s_y}, \qquad -1 \le r \le 1.$$

$|r| = 1$ exactly when the points lie on a straight line. Correlation measures only *linear* association: $y = x^2$ on values symmetric about zero gives $r = 0$ even though $y$ depends completely on $x$. A correlation also says nothing about cause and effect.

### Maximum likelihood estimation

Suppose the data $x_1, \ldots, x_n$ are independent draws from a model $p(x \mid \theta)$ with an unknown parameter $\theta$. The **likelihood** $L(\theta) = \prod_{i=1}^{n} p(x_i \mid \theta)$ measures how probable the observed data are for each value of $\theta$, and the **maximum likelihood estimate** (MLE) $\hat\theta$ is the value that makes it largest. A product of many numbers below $1$ quickly underflows to `0.0` in floating point, and the logarithm is increasing, so we maximise the **log-likelihood** $\ell(\theta) = \sum_{i=1}^{n} \log p(x_i \mid \theta)$ instead.

**Bernoulli.** With $k$ ones among $n$ observations,

$$\ell(p) = k \log p + (n - k) \log(1 - p), \qquad \ell'(p) = \frac{k}{p} - \frac{n - k}{1 - p} = 0 \;\Longrightarrow\; \hat{p} = \frac{k}{n}.$$

**Gaussian.** With mean $\mu$ and variance $\sigma^2$,

$$\ell(\mu, \sigma^2) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(x_i - \mu)^2.$$

Setting the derivative with respect to $\mu$ to zero gives $\hat\mu = \bar{x}$. Setting the derivative with respect to $\sigma^2$ to zero gives $-\frac{n}{2\sigma^2} + \frac{1}{2\sigma^4}\sum_i (x_i - \hat\mu)^2 = 0$, so

$$\hat\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2.$$

The MLE divides by $n$ because that is the value that maximises the likelihood; dividing by $n - 1$ serves a different goal, unbiasedness. The MLE is biased, $\mathbb{E}[\hat\sigma^2] = \frac{n-1}{n}\sigma^2$, so on average it underestimates the variance. The bias is large for tiny samples and negligible for big ones.

## Step-by-step implementation

### Step 1: describing data

`sorted` returns a new list, so `median` never reorders the caller's data. The sums use `math.fsum`, and both variances share one helper for the sum of squared deviations, so they differ only in the divisor.

In [ ]:
def mean(values):
    """The arithmetic mean of a non-empty list."""
    if len(values) == 0:
        raise ValueError("the mean of an empty list is undefined")
    return math.fsum(values) / len(values)


def median(values):
    """The middle value of the sorted data (the average of the two middle values for even n)."""
    if len(values) == 0:
        raise ValueError("the median of an empty list is undefined")
    ordered = sorted(values)  # a new list: the input stays as it was
    middle = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return ordered[middle]
    return (ordered[middle - 1] + ordered[middle]) / 2


def sum_of_squared_deviations(values):
    centre = mean(values)
    return math.fsum((x - centre) * (x - centre) for x in values)


def population_variance(values):
    """Average squared deviation from the mean: divide by n."""
    return sum_of_squared_deviations(values) / len(values)


def sample_variance(values):
    """Unbiased estimate of a larger population's variance: divide by n - 1."""
    if len(values) < 2:
        raise ValueError("the sample variance needs at least two values")
    return sum_of_squared_deviations(values) / (len(values) - 1)


def population_std(values):
    return math.sqrt(population_variance(values))


def sample_std(values):
    return math.sqrt(sample_variance(values))

In [ ]:
# Tests for Step 1
data = [2, 4, 4, 4, 5, 5, 7, 9]
assert math.isclose(mean(data), 5.0), "40 / 8 = 5"
assert math.isclose(median(data), 4.5), "the two middle values are 4 and 5"
assert median([3, 1, 2]) == 2, "with an odd count, the median is the middle value"
assert math.isclose(population_variance(data), 4.0), "squared deviations 9+1+1+1+0+0+4+16 = 32, and 32 / 8 = 4"
assert math.isclose(sample_variance(data), 32 / 7), "32 / (8 - 1)"
assert math.isclose(population_std(data), 2.0), "the square root of 4"
assert data == [2, 4, 4, 4, 5, 5, 7, 9], "median must not reorder its input"
assert math.isclose(mean([1, 2, 3, 4, 1000]), 202.0) and median([1, 2, 3, 4, 1000]) == 3, "an outlier drags the mean, not the median"
assert raises_value_error(mean, []), "the mean of nothing is undefined"
assert raises_value_error(sample_variance, [5]), "one value says nothing about spread"
assert issubclass(statistics.StatisticsError, ValueError), "statistics reports bad input with a subclass of ValueError"

rng = random.Random(3)
for _ in range(20):
    sample = [rng.uniform(-50, 50) for _ in range(rng.randint(2, 30))]
    assert math.isclose(mean(sample), statistics.fmean(sample), abs_tol=1e-12), "mean disagrees with statistics.fmean"
    assert math.isclose(median(sample), statistics.median(sample)), "median disagrees with statistics.median"
    assert math.isclose(population_variance(sample), statistics.pvariance(sample)), "population_variance disagrees with pvariance"
    assert math.isclose(sample_variance(sample), statistics.variance(sample)), "sample_variance disagrees with variance"
    assert math.isclose(population_std(sample), statistics.pstdev(sample)), "population_std disagrees with pstdev"
    assert math.isclose(sample_std(sample), statistics.stdev(sample)), "sample_std disagrees with stdev"
print("Step 1 tests passed: sample variance", sample_variance(data), "versus population variance", population_variance(data))

**Expected behaviour.** For `[2, 4, 4, 4, 5, 5, 7, 9]` the mean is $5$, the median $4.5$, the population variance $4$ and the standard deviation $2$, while the sample variance is $32/7 \approx 4.571$. One outlier of $1000$ moves the mean from about $2.5$ to $202$ but leaves the median at $3$. On 20 random samples all six functions agree with the `statistics` module.

### Step 2: probability distributions

Each function validates its parameters first. `binomial_pmf` returns $0$ for impossible counts instead of raising an error, because $P(K = 11)$ for ten trials is a perfectly good question whose answer is zero.

In [ ]:
def bernoulli_pmf(x, p):
    """P(X = x) for X ~ Bernoulli(p)."""
    if not 0 <= p <= 1:
        raise ValueError(f"p must lie in [0, 1], got {p}")
    if x == 1:
        return p
    if x == 0:
        return 1 - p
    return 0.0  # a Bernoulli variable only takes the values 0 and 1


def binomial_pmf(k, n, p):
    """P(K = k) for K ~ Binomial(n, p)."""
    if not 0 <= p <= 1:
        raise ValueError(f"p must lie in [0, 1], got {p}")
    if n < 0:
        raise ValueError(f"n must be non-negative, got {n}")
    if k < 0 or k > n:
        return 0.0
    return math.comb(n, k) * p ** k * (1 - p) ** (n - k)


def normal_pdf(x, mu=0.0, sigma=1.0):
    """The normal density f(x | mu, sigma)."""
    if sigma <= 0:
        raise ValueError(f"sigma must be positive, got {sigma}")
    z = (x - mu) / sigma
    return math.exp(-0.5 * z * z) / (sigma * math.sqrt(2 * math.pi))

In [ ]:
# Tests for Step 2
assert math.isclose(bernoulli_pmf(1, 0.3), 0.3) and math.isclose(bernoulli_pmf(0, 0.3), 0.7), "P(1) = p and P(0) = 1 - p"
assert math.isclose(binomial_pmf(2, 4, 0.5), 0.375), "C(4, 2) / 2^4 = 6 / 16"
assert binomial_pmf(5, 4, 0.5) == 0.0 and binomial_pmf(-1, 4, 0.5) == 0.0, "impossible counts have probability 0"
for n, p in [(1, 0.3), (10, 0.5), (20, 0.13)]:
    probabilities = [binomial_pmf(k, n, p) for k in range(n + 1)]
    assert math.isclose(math.fsum(probabilities), 1.0), f"Binomial({n}, {p}) probabilities must sum to 1"
    assert math.isclose(math.fsum(k * q for k, q in enumerate(probabilities)), n * p), "the mean of a binomial is np"
assert math.isclose(normal_pdf(0.0), 1 / math.sqrt(2 * math.pi)), "the standard normal peaks at 1 / sqrt(2 pi)"
for x in (-2.5, -1.0, 0.0, 0.7, 3.0):
    assert math.isclose(normal_pdf(x, 1.0, 2.0), statistics.NormalDist(1.0, 2.0).pdf(x)), "normal_pdf disagrees with NormalDist"
    assert math.isclose(normal_pdf(1.0 + x, 1.0, 2.0), normal_pdf(1.0 - x, 1.0, 2.0)), "the density is symmetric about its mean"
assert normal_pdf(0.0, 0.0, 0.1) > 1, "a narrow density is taller than 1"
width = 0.001
area = math.fsum(normal_pdf(-8 + (i + 0.5) * width) * width for i in range(16_000))
assert math.isclose(area, 1.0, rel_tol=1e-9), "the area under the density should be 1"
assert raises_value_error(normal_pdf, 0.0, 0.0, 0.0), "sigma must be positive"
assert raises_value_error(binomial_pmf, 1, 4, 1.5), "p must be a probability"
print("Step 2 tests passed: area under the standard normal from -8 to 8 =", area)

**Expected behaviour.** $P(K = 2) = \binom{4}{2}/2^4 = 6/16 = 0.375$ for four fair coin flips. Every binomial distribution's probabilities sum to $1$ and its mean is $np$. The normal density matches `statistics.NormalDist`, is symmetric about its mean, peaks at $1/\sqrt{2\pi} \approx 0.399$ for the standard normal and exceeds $1$ when $\sigma = 0.1$. A midpoint-rule sum over $[-8, 8]$ confirms that the total area is $1$: the printed value differs from $1$ only around the fifteenth decimal place.

### Step 3: sampling with a seeded generator

`bernoulli_sample` turns one uniform number from `rng.random()` into a coin flip with probability $p$; the other two functions build on it. All three take the generator as an argument.

In [ ]:
def bernoulli_sample(p, rng):
    """One draw from Bernoulli(p): 1 with probability p, otherwise 0."""
    return 1 if rng.random() < p else 0


def bernoulli_samples(p, n, rng):
    """n independent Bernoulli(p) draws."""
    return [bernoulli_sample(p, rng) for _ in range(n)]


def binomial_sample(n, p, rng):
    """One draw from Binomial(n, p): the number of successes in n trials."""
    return sum(bernoulli_samples(p, n, rng))


rng = random.Random(42)
print(f"{'n':>8}  {'sample mean':>11}  {'error':>8}  {'standard error':>14}")
for n in (10, 100, 1_000, 10_000, 100_000):
    sample_mean = mean(bernoulli_samples(0.3, n, rng))
    print(f"{n:>8,}  {sample_mean:>11.4f}  {sample_mean - 0.3:>+8.4f}  {math.sqrt(0.3 * 0.7 / n):>14.4f}")

In [ ]:
# Tests for Step 3
first = bernoulli_samples(0.3, 20, random.Random(42))
second = bernoulli_samples(0.3, 20, random.Random(42))
assert first == second, "the same seed must reproduce the same samples"
assert set(first) <= {0, 1}, "Bernoulli samples are 0 or 1"
assert bernoulli_samples(0.3, 20, random.Random(7)) != first, "a different seed gives a different sequence"
assert abs(mean(bernoulli_samples(0.3, 100_000, random.Random(0))) - 0.3) < 0.01, "100,000 draws should give a mean within 0.01 of p"

rng = random.Random(1)
trials = 20_000
counts = [0] * 11
for _ in range(trials):
    counts[binomial_sample(10, 0.5, rng)] += 1
for k in range(11):
    assert abs(counts[k] / trials - binomial_pmf(k, 10, 0.5)) < 0.01, f"the simulated P(K = {k}) is far from the pmf"

rng = random.Random(5)
draws = [rng.gauss(1.0, 2.0) for _ in range(50_000)]
assert abs(mean(draws) - 1.0) < 0.05, "the sample mean of N(1, 2^2) draws should be close to 1"
assert abs(sample_std(draws) - 2.0) < 0.05, "their sample standard deviation should be close to 2"
print("Step 3 tests passed: simulated P(K = 5) =", counts[5] / trials, "versus", binomial_pmf(5, 10, 0.5))

**Expected behaviour.** In the table, the error of the sample mean shrinks roughly like the standard error, from $0.2$ at $n = 10$ to about $0.001$ at $n = 100{,}000$. The exact numbers depend on the seed, but the trend does not. Two generators created with the same seed produce identical samples. Over 20,000 simulated experiments of ten fair coin flips, the frequency of every count from 0 to 10 lies within $0.01$ of the binomial pmf; for example, five heads turn up in about 25% of the experiments, against the exact $252/1024 \approx 24.6\%$. Gaussian draws reproduce the requested mean and standard deviation.

### Step 4: conditional probability and Bayes' theorem

For two fair dice every one of the 36 ordered outcomes is equally likely, so probabilities are ratios of counts. `Fraction` keeps them exact. `bayes_posterior` applies Bayes' theorem to a screening test, and `simulate_screening` checks the formula against simulated people.

In [ ]:
from fractions import Fraction
from itertools import product

TWO_DICE = list(product(range(1, 7), repeat=2))  # 36 equally likely (first, second) pairs


def probability(event, outcomes):
    """P(event) for equally likely outcomes, as an exact fraction."""
    hits = sum(1 for outcome in outcomes if event(outcome))
    return Fraction(hits, len(outcomes))


def conditional_probability(event, given, outcomes):
    """P(event | given) for equally likely outcomes."""
    remaining = [outcome for outcome in outcomes if given(outcome)]
    if not remaining:
        raise ValueError("cannot condition on an event that never happens")
    return probability(event, remaining)


def bayes_posterior(prior, sensitivity, false_positive_rate):
    """P(condition | positive test) by Bayes' theorem.

    prior               P(condition) before testing, the base rate
    sensitivity         P(positive | condition)
    false_positive_rate P(positive | no condition)
    """
    for name, value in [("prior", prior), ("sensitivity", sensitivity), ("false_positive_rate", false_positive_rate)]:
        if not 0 <= value <= 1:
            raise ValueError(f"{name} must lie in [0, 1], got {value}")
    evidence = sensitivity * prior + false_positive_rate * (1 - prior)  # P(positive)
    if evidence == 0:
        raise ValueError("a positive result is impossible with these numbers")
    return sensitivity * prior / evidence


def simulate_screening(prior, sensitivity, false_positive_rate, people, rng):
    """Simulate a screening programme; return the share of positives who have the condition."""
    true_positives = false_positives = 0
    for _ in range(people):
        has_condition = rng.random() < prior
        tests_positive = rng.random() < (sensitivity if has_condition else false_positive_rate)
        if tests_positive and has_condition:
            true_positives += 1
        elif tests_positive:
            false_positives += 1
    return true_positives / (true_positives + false_positives)

In [ ]:
# Tests for Step 4
def total_is_8(outcome):
    return sum(outcome) == 8


def first_is_even(outcome):
    return outcome[0] % 2 == 0


assert probability(total_is_8, TWO_DICE) == Fraction(5, 36), "(2,6), (3,5), (4,4), (5,3), (6,2)"
assert conditional_probability(total_is_8, first_is_even, TWO_DICE) == Fraction(1, 6), "3 of the 18 outcomes with an even first die"
assert conditional_probability(first_is_even, total_is_8, TWO_DICE) == Fraction(3, 5), "3 of the 5 ways to make 8"
both = probability(lambda o: total_is_8(o) and first_is_even(o), TWO_DICE)
assert conditional_probability(total_is_8, first_is_even, TWO_DICE) == both / probability(first_is_even, TWO_DICE), "P(A | B) = P(A and B) / P(B)"
assert conditional_probability(lambda o: o[0] == 6, lambda o: o[1] == 6, TWO_DICE) == Fraction(1, 6), "independent dice: the second tells us nothing about the first"
assert raises_value_error(conditional_probability, total_is_8, lambda o: sum(o) == 13, TWO_DICE), "a total of 13 never happens"

screening_posterior = bayes_posterior(0.01, 0.95, 0.05)
assert math.isclose(screening_posterior, 95 / 590), "95 true positives out of 590 positives per 10,000 people"
assert bayes_posterior(Fraction(1, 100), Fraction(95, 100), Fraction(5, 100)) == Fraction(19, 118), "the exact posterior is 19/118"
assert math.isclose(bayes_posterior(0.5, 0.9, 0.1), 0.9), "with a 50% prior and a symmetric test, the posterior equals the sensitivity"
simulated = simulate_screening(0.01, 0.95, 0.05, 200_000, random.Random(2024))
assert abs(simulated - screening_posterior) < 0.02, "the simulation should agree with Bayes' theorem"
print(f"P(condition | positive) = {screening_posterior:.4f} by Bayes' theorem and {simulated:.4f} by simulation")

**Expected behaviour.** With two dice, $P(\text{total} = 8) = 5/36$, but $P(\text{total} = 8 \mid \text{first die even}) = 3/18 = 1/6$ and $P(\text{first die even} \mid \text{total} = 8) = 3/5$: reversing a condition changes the answer. For the screening test (1% prevalence, 95% sensitivity, 5% false-positive rate), think of 10,000 people. Of the 100 who have the condition, 95 test positive; of the 9,900 who do not, 495 also test positive. Only $95/590 \approx 16.1\%$ of the positives have the condition, and 200,000 simulated people agree to within about one percentage point.

### Step 5: covariance and correlation

`covariance` divides by $n - 1$, like `statistics.covariance`. In `correlation` that divisor cancels, because it appears in the covariance and in both standard deviations.

In [ ]:
def covariance(x, y):
    """Sample covariance of paired values (divides by n - 1)."""
    if len(x) != len(y):
        raise ValueError(f"x and y must have the same length, got {len(x)} and {len(y)}")
    if len(x) < 2:
        raise ValueError("covariance needs at least two pairs")
    mean_x, mean_y = mean(x), mean(y)
    return math.fsum((xi - mean_x) * (yi - mean_y) for xi, yi in zip(x, y)) / (len(x) - 1)


def correlation(x, y):
    """Pearson's correlation coefficient r."""
    cov = covariance(x, y)  # validates the inputs
    spread_x, spread_y = sample_std(x), sample_std(y)
    if spread_x == 0 or spread_y == 0:
        raise ValueError("correlation is undefined when one variable is constant")
    return cov / (spread_x * spread_y)

In [ ]:
# Tests for Step 5
x = [1, 2, 3, 4, 5]
y = [2, 4, 5, 4, 5]
assert math.isclose(covariance(x, y), 1.5), "products of deviations 4+0+0+0+2 = 6, and 6 / 4 = 1.5"
assert math.isclose(correlation(x, y), 1.5 / math.sqrt(2.5 * 1.5)), "r = 1.5 / sqrt(2.5 * 1.5)"
assert math.isclose(covariance(x, y), statistics.covariance(x, y)), "covariance disagrees with statistics.covariance"
assert math.isclose(correlation(x, y), statistics.correlation(x, y)), "correlation disagrees with statistics.correlation"
assert math.isclose(correlation(x, [3 * xi - 2 for xi in x]), 1.0), "an increasing straight line gives r = 1"
assert math.isclose(correlation(x, [7 - 2 * xi for xi in x]), -1.0), "a decreasing straight line gives r = -1"
symmetric = [-2, -1, 0, 1, 2]
assert math.isclose(correlation(symmetric, [s * s for s in symmetric]), 0.0, abs_tol=1e-12), "y = x^2 is dependent but uncorrelated"
assert math.isclose(correlation(x, [10 * yi + 3 for yi in y]), correlation(x, y)), "r ignores changes of units"
assert raises_value_error(correlation, [1, 2, 3], [4, 4, 4]), "a constant variable has no correlation"
assert raises_value_error(covariance, [1, 2], [1, 2, 3]), "x and y must be paired"
print("Step 5 tests passed: r =", correlation(x, y))

**Expected behaviour.** For the five pairs, the covariance is $6/4 = 1.5$ and $r = 1.5/\sqrt{2.5 \times 1.5} \approx 0.775$, matching the `statistics` module. Points on an increasing line give $r = 1$ and points on a decreasing line give $r = -1$. Rescaling $y$ to new units leaves $r$ unchanged, and $y = x^2$ on symmetric values gives $r = 0$ even though $y$ is completely determined by $x$.

### Step 6: maximum likelihood estimation

The log-likelihood functions translate the formulas from the theory. `bernoulli_log_likelihood` returns $-\infty$ when the data contain an outcome that the parameter makes impossible, such as a `1` when $p = 0$, instead of trying to compute $\log 0$. The tests confirm that the closed-form estimates really do maximise the log-likelihood.

In [ ]:
def bernoulli_log_likelihood(p, data):
    """log L(p) = sum of log P(x_i | p) for 0/1 data; -inf if some observation is impossible."""
    if not 0 <= p <= 1:
        raise ValueError(f"p must lie in [0, 1], got {p}")
    total = 0.0
    for x in data:
        probability_of_x = p if x == 1 else 1 - p
        if probability_of_x == 0:
            return -math.inf  # log(0): this p cannot have produced the data
        total += math.log(probability_of_x)
    return total


def bernoulli_mle(data):
    """The p that maximises the Bernoulli likelihood: the proportion of ones."""
    if len(data) == 0:
        raise ValueError("cannot estimate p from no data")
    if any(x not in (0, 1) for x in data):
        raise ValueError("Bernoulli data must be 0 or 1")
    return sum(data) / len(data)


def gaussian_log_likelihood(mu, variance, data):
    """log L(mu, variance) for independent normal observations."""
    if variance <= 0:
        raise ValueError(f"the variance must be positive, got {variance}")
    squared = math.fsum((x - mu) * (x - mu) for x in data)
    return -0.5 * len(data) * math.log(2 * math.pi * variance) - squared / (2 * variance)


def gaussian_mle(data):
    """(mu_hat, variance_hat): the sample mean and the variance that divides by n."""
    if len(data) == 0:
        raise ValueError("cannot estimate a Gaussian from no data")
    return mean(data), population_variance(data)

In [ ]:
# Tests for Step 6
coin = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1]  # 7 ones in 10 flips
p_hat = bernoulli_mle(coin)
assert math.isclose(p_hat, 0.7), "the MLE is the proportion of ones, 7 / 10"
grid = [i / 100 for i in range(1, 100)]
best_on_grid = max(grid, key=lambda p: bernoulli_log_likelihood(p, coin))
assert math.isclose(best_on_grid, p_hat), "a grid search should find its maximum at k / n"
assert bernoulli_log_likelihood(0.0, coin) == -math.inf, "p = 0 cannot produce a 1"
assert math.isclose(bernoulli_log_likelihood(1.0, [1, 1, 1]), 0.0, abs_tol=1e-12), "p = 1 explains all-ones data with probability 1"
assert raises_value_error(bernoulli_mle, [1, 2]), "Bernoulli data must be 0 or 1"

product_of_probabilities = 1.0
for _ in range(2_000):
    product_of_probabilities *= 0.5
assert product_of_probabilities == 0.0, "0.5 ** 2000 underflows to 0.0"
assert math.isclose(bernoulli_log_likelihood(0.5, [0, 1] * 1_000), 2_000 * math.log(0.5)), "the log-likelihood stays finite"

rng = random.Random(11)
heights = [rng.gauss(170.0, 8.0) for _ in range(1_000)]
mu_hat, variance_hat = gaussian_mle(heights)
assert math.isclose(mu_hat, statistics.fmean(heights)), "mu_hat is the sample mean"
assert math.isclose(variance_hat, statistics.pvariance(heights)), "variance_hat divides by n, like pvariance"
assert abs(mu_hat - 170.0) < 1.5 and abs(math.sqrt(variance_hat) - 8.0) < 1.0, "the estimates should be near the true values"
best = gaussian_log_likelihood(mu_hat, variance_hat, heights)
for mu, variance in [(mu_hat + 0.5, variance_hat), (mu_hat - 0.5, variance_hat),
                     (mu_hat, variance_hat * 1.1), (mu_hat, variance_hat * 0.9),
                     (mu_hat, sample_variance(heights))]:
    assert gaussian_log_likelihood(mu, variance, heights) < best, "the MLE should beat every nearby alternative"

rng = random.Random(7)
repeats, n = 20_000, 5
mle_total = unbiased_total = 0.0
for _ in range(repeats):
    small_sample = [rng.gauss(0.0, 1.0) for _ in range(n)]
    mle_total += gaussian_mle(small_sample)[1]
    unbiased_total += sample_variance(small_sample)
average_mle, average_unbiased = mle_total / repeats, unbiased_total / repeats
assert abs(average_mle - (n - 1) / n) < 0.03, "on average the MLE should be (n - 1)/n = 0.8 of the true variance"
assert abs(average_unbiased - 1.0) < 0.03, "on average the sample variance should match the true variance, 1"
print(f"Average over {repeats:,} samples of size {n}: MLE {average_mle:.3f}, sample variance {average_unbiased:.3f}, true variance 1")

**Expected behaviour.** Seven heads in ten flips give $\hat{p} = 0.7$, and a grid search over $p = 0.01, 0.02, \ldots, 0.99$ peaks at the same value. Multiplying two thousand probabilities of $0.5$ underflows to exactly `0.0`, while the log-likelihood stays a finite $2000 \log 0.5 \approx -1386.3$. For 1,000 simulated heights the MLE equals `statistics.fmean` and `statistics.pvariance`, lies close to the true values $\mu = 170$ and $\sigma = 8$, and scores a higher log-likelihood than every nearby alternative, including the sample variance $s^2$. Averaged over 20,000 samples of size 5, $\hat\sigma^2$ comes out near $0.8$ and $s^2$ near $1.0$: the MLE is biased by the factor $(n-1)/n$, exactly as the theory predicts.

### Optional comparison: NumPy

*Optional: you can skip this cell, and nothing later depends on it.* NumPy's `var` and `std` divide by $n$ unless you pass `ddof=1` ("delta degrees of freedom"), which makes them divide by $n - 1$. Mixing up the two conventions is a common source of small, confusing discrepancies. If NumPy is not installed, the cell prints a note instead of failing.

In [ ]:
try:
    import numpy as np
except ImportError:
    print("NumPy is not available here; skipping this optional comparison.")
else:
    example = [2, 4, 4, 4, 5, 5, 7, 9]
    print("np.var(example)         =", float(np.var(example)), "  population_variance =", population_variance(example))
    print("np.var(example, ddof=1) =", float(np.var(example, ddof=1)), "  sample_variance     =", sample_variance(example))
    pairs_x, pairs_y = [1, 2, 3, 4, 5], [2, 4, 5, 4, 5]
    print("np.corrcoef(x, y)[0, 1] =", float(np.corrcoef(pairs_x, pairs_y)[0, 1]), "  correlation =", correlation(pairs_x, pairs_y))

**Expected behaviour.** `np.var` gives `4.0`, the population variance, and `np.var(..., ddof=1)` gives about `4.571`, the sample variance. Both correlations print about `0.7746`.

### Optional visualisation: samples against formulas

*Optional: you can skip this cell, and nothing later depends on it.* The left panel compares a histogram of 10,000 standard normal samples with `normal_pdf`; the right panel compares simulated binomial frequencies with `binomial_pmf`. If Matplotlib is not installed, the cell prints a note instead of failing.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Matplotlib is not available here; skipping this optional plot.")
else:
    plot_rng = random.Random(11)
    normal_draws = [plot_rng.gauss(0.0, 1.0) for _ in range(10_000)]
    grid_points = [-4 + 8 * i / 200 for i in range(201)]
    counts_for_plot = [0] * 11
    for _ in range(5_000):
        counts_for_plot[binomial_sample(10, 0.5, plot_rng)] += 1

    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.3))
    left.hist(normal_draws, bins=40, density=True, color="#2a78d6", edgecolor="white", linewidth=0.8,
              label="10,000 samples")
    left.plot(grid_points, [normal_pdf(g) for g in grid_points], color="#eb6834", linewidth=2, label="normal_pdf(x)")
    left.set_title("Standard normal: samples and density")
    left.set_xlabel("x")
    left.set_ylabel("density")

    ks = list(range(11))
    right.bar([k - 0.2 for k in ks], [c / 5_000 for c in counts_for_plot], width=0.38, color="#2a78d6",
              label="simulated frequency")
    right.bar([k + 0.2 for k in ks], [binomial_pmf(k, 10, 0.5) for k in ks], width=0.38, color="#eb6834",
              label="binomial_pmf(k, 10, 0.5)")
    right.set_title("Binomial(10, 0.5): simulation and formula")
    right.set_xlabel("number of successes k")
    right.set_ylabel("probability")
    right.set_xticks(ks)

    for axis in (left, right):
        axis.grid(axis="y", color="#e4e4e0", linewidth=0.8)
        axis.set_axisbelow(True)
        axis.spines[["top", "right"]].set_visible(False)
        axis.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    fig.tight_layout()
    plt.show()

**Expected behaviour.** The histogram follows the bell-shaped curve, and each simulated bar stands close to its formula bar, with the tallest pair at $k = 5$.

## Exercises

Each exercise gives you a function with a docstring, a `TODO` comment and a `raise NotImplementedError(...)` line. Replace that line with your own code, run the cell, then run the `check` cell below it. Until you do, `check` prints `not implemented yet`, so the notebook still runs from top to bottom. You may use any function from the implementation steps. The reference solutions are at the end of the notebook; try each exercise before you look.

### Exercise 1: describing and relating features

* `standardise(values)` returns the **z-scores** $z_i = (x_i - \bar{x}) / \sigma_{\text{pop}}$, using the population standard deviation as scikit-learn's `StandardScaler` does. The result has mean $0$ and standard deviation $1$. Raise `ValueError` for an empty list and for constant data, which has no spread to divide by.
* `covariance_matrix(rows)` takes a dataset as a list of rows (one example per row, one feature per column) and returns the $d \times d$ matrix whose entry $(j, k)$ is the sample covariance of features $j$ and $k$. The diagonal holds each feature's sample variance. Raise `ValueError` for fewer than two rows, and for empty or ragged rows.

In [ ]:
def standardise(values):
    """z-scores (x - mean) / population standard deviation."""
    # TODO: reject empty and constant data, then rescale every value.
    raise NotImplementedError("Exercise 1a: implement standardise")


def covariance_matrix(rows):
    """The d x d sample covariance matrix of a dataset given as rows."""
    # TODO: validate the rows, extract each column, then fill in covariance(column_j, column_k).
    raise NotImplementedError("Exercise 1b: implement covariance_matrix")

In [ ]:
STANDARDISE_CASES = [
    (([2, 4, 4, 4, 5, 5, 7, 9],), [-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0]),
    (([10, 20],), [-1.0, 1.0]),
    (([3, 3, 3],), ValueError),
    (([],), ValueError),
]
COVARIANCE_MATRIX_CASES = [
    (([[1, 2], [2, 4], [3, 5], [4, 4], [5, 5]],), [[2.5, 1.5], [1.5, 1.5]]),
    (([[1, 0, 2], [3, 0, 4], [5, 0, 0]],), [[4.0, 0.0, -2.0], [0.0, 0.0, 0.0], [-2.0, 0.0, 4.0]]),
    (([[1, 2]],), ValueError),  # one row cannot show any spread
    (([[1, 2], [3]],), ValueError),  # ragged rows
]
check("standardise", standardise, STANDARDISE_CASES)
check("covariance_matrix", covariance_matrix, COVARIANCE_MATRIX_CASES)

### Exercise 2: distributions and sampling

* `binomial_cdf(k, n, p)` returns the cumulative probability $P(K \le k) = \sum_{i=0}^{k} P(K = i)$. Return `0.0` when $k < 0$ and `1.0` when $k \ge n$. Raise `ValueError` when $p$ is outside $[0, 1]$, even for those easy cases.
* `estimate_dice_probability(target, trials, rng)` simulates `trials` rolls of two fair dice using the generator `rng`, and returns the fraction of rolls whose total equals `target`. Raise `ValueError` when `trials` is less than 1. The tests compare your estimate with the exact probability, with a tolerance of $0.015$ (more than five standard errors), so any correct method of rolling a die passes.

In [ ]:
def binomial_cdf(k, n, p):
    """P(K <= k) for K ~ Binomial(n, p)."""
    # TODO: validate p, handle k < 0 and k >= n, then add up binomial_pmf(i, n, p).
    raise NotImplementedError("Exercise 2a: implement binomial_cdf")


def estimate_dice_probability(target, trials, rng):
    """The fraction of `trials` simulated rolls of two dice whose total is `target`."""
    # TODO: validate trials, roll two dice per trial with rng.randint(1, 6), count the hits.
    raise NotImplementedError("Exercise 2b: implement estimate_dice_probability")

In [ ]:
BINOMIAL_CDF_CASES = [
    ((2, 4, 0.5), 0.6875),  # (1 + 4 + 6) / 16
    ((0, 4, 0.5), 0.0625),
    ((4, 4, 0.5), 1.0),
    ((-1, 4, 0.5), 0.0),
    ((10, 4, 0.5), 1.0),
    ((3, 10, 0.3), 0.6496107184),
    ((2, 4, 1.5), ValueError),
]
DICE_CASES = [
    ((7, 20_000, random.Random(1)), 6 / 36),
    ((2, 20_000, random.Random(2)), 1 / 36),
    ((10, 20_000, random.Random(3)), 3 / 36),
    ((13, 1_000, random.Random(4)), 0.0),  # impossible with two dice
    ((7, 0, random.Random(5)), ValueError),
]
check("binomial_cdf", binomial_cdf, BINOMIAL_CDF_CASES)
check("estimate_dice_probability", estimate_dice_probability, DICE_CASES, abs_tol=0.015)

### Exercise 3: Bayes' theorem

* `posterior(priors, likelihoods)` handles several mutually exclusive hypotheses $H_1, \ldots, H_m$ at once. Given their priors $P(H_j)$, which must sum to $1$, and the likelihoods $P(E \mid H_j)$ of the observed evidence, return the list of posteriors

$$P(H_j \mid E) = \frac{P(E \mid H_j)\, P(H_j)}{\sum_{i} P(E \mid H_i)\, P(H_i)}.$$

Raise `ValueError` for empty or unequal lists, for priors that do not sum to $1$ (compare with `math.isclose`), and when the evidence has probability $0$.
* `repeated_test_posterior(prior, sensitivity, false_positive_rate, positives)` returns the probability of the condition after `positives` independent positive test results. Apply Bayes' theorem once per result, using each posterior as the prior for the next test. With `positives = 0`, return the prior. Raise `ValueError` for a negative count.

In [ ]:
def posterior(priors, likelihoods):
    """Posterior probabilities of mutually exclusive hypotheses after seeing the evidence."""
    # TODO: validate, multiply each prior by its likelihood, then divide by the total.
    raise NotImplementedError("Exercise 3a: implement posterior")


def repeated_test_posterior(prior, sensitivity, false_positive_rate, positives):
    """P(condition) after `positives` independent positive results."""
    # TODO: start from the prior and apply bayes_posterior once per positive result.
    raise NotImplementedError("Exercise 3b: implement repeated_test_posterior")

In [ ]:
POSTERIOR_CASES = [
    (([0.5, 0.5], [0.8, 0.2]), [0.8, 0.2]),
    (([0.2, 0.8], [0.4, 0.01]), [10 / 11, 1 / 11]),  # spam filter: 0.08 against 0.008
    (([0.01, 0.99], [0.95, 0.05]), [95 / 590, 495 / 590]),  # the screening test from Step 4
    (([0.5, 0.3, 0.2], [0.1, 0.5, 0.9]), [5 / 38, 15 / 38, 18 / 38]),
    (([0.5, 0.6], [0.1, 0.2]), ValueError),  # the priors sum to 1.1
    (([0.5, 0.5], [0.0, 0.0]), ValueError),  # the evidence is impossible
    (([0.5, 0.5], [0.1]), ValueError),
]
REPEATED_TEST_CASES = [
    ((0.01, 0.95, 0.05, 0), 0.01),
    ((0.01, 0.95, 0.05, 1), 19 / 118),
    ((0.01, 0.95, 0.05, 2), 361 / 460),
    ((0.01, 0.95, 0.05, 3), 6859 / 6958),
    ((0.5, 0.9, 0.1, 1), 0.9),
    ((0.01, 0.95, 0.05, -1), ValueError),
]
check("posterior", posterior, POSTERIOR_CASES)
check("repeated_test_posterior", repeated_test_posterior, REPEATED_TEST_CASES)

### Exercise 4: maximum likelihood for waiting times

The time between independent random events (such as requests reaching a server) is often modelled by an **exponential distribution** with rate $\lambda > 0$, whose density is $p(x \mid \lambda) = \lambda e^{-\lambda x}$ for $x \ge 0$. For observed waiting times $x_1, \ldots, x_n$,

$$\ell(\lambda) = \sum_{i=1}^{n} \log\left(\lambda e^{-\lambda x_i}\right) = n \log \lambda - \lambda \sum_{i=1}^{n} x_i.$$

* `exponential_log_likelihood(rate, data)` returns $\ell(\lambda)$. Raise `ValueError` if the rate is not positive, the data are empty, or any waiting time is negative.
* `exponential_mle(data)` returns $\hat\lambda$. Derive it yourself first: differentiate $\ell(\lambda)$, set the derivative to zero and solve for $\lambda$. Raise `ValueError` for empty data, a negative waiting time, or data that are all zero (the estimate would be infinite).

In [ ]:
def exponential_log_likelihood(rate, data):
    """n log(rate) - rate * sum(data) for non-negative waiting times."""
    # TODO: validate the rate and the data, then apply the formula.
    raise NotImplementedError("Exercise 4a: implement exponential_log_likelihood")


def exponential_mle(data):
    """The rate that maximises the exponential log-likelihood."""
    # TODO: solve d/d(rate) [n log(rate) - rate * sum(data)] = 0 on paper, then code the answer.
    raise NotImplementedError("Exercise 4b: implement exponential_mle")

In [ ]:
EXPONENTIAL_LL_CASES = [
    ((1.0, [1.0, 2.0, 3.0]), -6.0),  # 3 log 1 - 1 * 6
    ((0.5, [1.0, 2.0, 3.0]), 3 * math.log(0.5) - 3.0),
    ((2.0, [0.5]), math.log(2.0) - 1.0),
    ((0.0, [1.0]), ValueError),
    ((1.0, [-1.0]), ValueError),
    ((1.0, []), ValueError),
]
EXPONENTIAL_MLE_CASES = [
    (([1.0, 2.0, 3.0],), 0.5),
    (([0.25, 0.25, 0.5],), 3.0),
    (([4.0],), 0.25),
    (([],), ValueError),
    (([0.0, 0.0],), ValueError),
    (([1.0, -2.0],), ValueError),
]
check("exponential_log_likelihood", exponential_log_likelihood, EXPONENTIAL_LL_CASES)
check("exponential_mle", exponential_mle, EXPONENTIAL_MLE_CASES)

## Common errors and debugging guidance

| Symptom | Likely cause | What to do |
|---|---|---|
| your variance differs slightly from a library's | one divides by $n$ and the other by $n - 1$ | `statistics.pvariance` and NumPy's default divide by $n$; `statistics.variance`, `ddof=1` and pandas' default divide by $n - 1$ |
| a list is in a different order after computing a median | `values.sort()` sorts the caller's list in place | use `sorted(values)`, which returns a new list |
| the "random" number is the same on every pass of a loop | `random.Random(42)` re-created inside the loop | create one generator before the loop and pass it in |
| results change when unrelated code is added | the shared generator behind `random.random()` is used everywhere | give each experiment its own `random.Random(seed)` |
| a test of random code fails after an upgrade | it compared exact sampled values | test statistical properties with a tolerance, or compare two runs that share a seed |
| a 99%-accurate test seems to be wrong most of the time | the base rate was ignored, confusing $P(E \mid H)$ with $P(H \mid E)$ | apply Bayes' theorem, or count natural frequencies out of 10,000 people |
| a "probability" above 1 | a density value was read as a probability | probabilities are areas under a density |
| a likelihood of exactly `0.0` | the product of many probabilities underflowed | add log-probabilities instead |
| `ValueError` from `math.log` | the logarithm of $0$ for an impossible observation | handle it explicitly, for example by returning `-math.inf` |
| $r \approx 0$, so you conclude "no relationship" | correlation only detects straight-line relationships | plot the data; $y = x^2$ has $r = 0$ |

**Debugging habits for statistics and probability**

1. **Start with data you can do in your head.** `[2, 4, 4, 4, 5, 5, 7, 9]` has mean $5$ and population variance $4$, so it immediately shows which divisor a function uses.
2. **Compare with the standard library.** The `statistics` module (`fmean`, `median`, `pvariance`, `variance`, `covariance`, `correlation`, `NormalDist`) is a trustworthy reference.
3. **Check that probabilities add up.** A pmf summed over all values, or a posterior summed over all hypotheses, must give $1$ (use `math.fsum` and `math.isclose`).
4. **Simulate the formula.** If a seeded simulation with many draws disagrees with your formula by many standard errors, one of them is wrong.
5. **Fix every seed.** A bug that appears only for some random draws is far easier to find when the same draws come back every time.

## Summary

* The mean is pulled by outliers and the median is not. The population variance divides by $n$, and the sample variance divides by $n - 1$ to be unbiased.
* The Bernoulli and binomial pmfs give probabilities, $\binom{n}{k} p^k (1-p)^{n-k}$, while the normal pdf gives a density, which can exceed $1$; probabilities are areas under a density.
* `random.Random(seed)`, passed into functions as an argument, makes random experiments reproducible. Sample averages approach expectations, with an error that shrinks like $1/\sqrt{n}$.
* $P(A \mid B) = P(A \cap B)/P(B)$, and Bayes' theorem reverses a condition: with a rare condition, most positive results can be false positives.
* Covariance and correlation measure linear co-movement; $r$ ignores units, lies in $[-1, 1]$ and misses non-linear relationships.
* Maximum likelihood chooses the parameters that make the data most probable: $\hat{p} = k/n$, $\hat\mu = \bar{x}$ and $\hat\sigma^2 = \frac{1}{n}\sum_i (x_i - \bar{x})^2$. Work with log-likelihoods to avoid underflow.

## Solutions

Try every exercise before you read this section. Each reference solution has a `_solution` suffix, so it never replaces your own function. Each one runs through the same `check` cases with `strict=True`, which turns `not implemented yet` into a failure, so passing here shows that the tests themselves are correct.

In [ ]:
def standardise_solution(values):
    if len(values) == 0:
        raise ValueError("cannot standardise an empty list")
    spread = population_std(values)
    if spread == 0:
        raise ValueError("constant data have no spread to divide by")
    centre = mean(values)
    return [(x - centre) / spread for x in values]


def covariance_matrix_solution(rows):
    if len(rows) < 2:
        raise ValueError("a covariance matrix needs at least two rows")
    width = len(rows[0])
    if width == 0 or any(len(row) != width for row in rows):
        raise ValueError("every row must have the same, non-zero number of features")
    columns = [[row[j] for row in rows] for j in range(width)]
    return [[covariance(columns[j], columns[k]) for k in range(width)] for j in range(width)]


check("standardise_solution", standardise_solution, STANDARDISE_CASES, strict=True)
check("covariance_matrix_solution", covariance_matrix_solution, COVARIANCE_MATRIX_CASES, strict=True)

In [ ]:
def binomial_cdf_solution(k, n, p):
    if not 0 <= p <= 1:
        raise ValueError(f"p must lie in [0, 1], got {p}")
    if k < 0:
        return 0.0
    if k >= n:
        return 1.0
    return math.fsum(binomial_pmf(i, n, p) for i in range(k + 1))


def estimate_dice_probability_solution(target, trials, rng):
    if trials < 1:
        raise ValueError(f"need at least one trial, got {trials}")
    hits = 0
    for _ in range(trials):
        if rng.randint(1, 6) + rng.randint(1, 6) == target:
            hits += 1
    return hits / trials


check("binomial_cdf_solution", binomial_cdf_solution, BINOMIAL_CDF_CASES, strict=True)
check("estimate_dice_probability_solution", estimate_dice_probability_solution, DICE_CASES, abs_tol=0.015, strict=True)

In [ ]:
def posterior_solution(priors, likelihoods):
    if len(priors) == 0 or len(priors) != len(likelihoods):
        raise ValueError("need one likelihood for each prior, and at least one hypothesis")
    if not math.isclose(math.fsum(priors), 1.0):
        raise ValueError(f"the priors must sum to 1, got {math.fsum(priors)}")
    joint = [prior * likelihood for prior, likelihood in zip(priors, likelihoods)]  # P(H_j and E)
    evidence = math.fsum(joint)  # P(E), by the law of total probability
    if evidence == 0:
        raise ValueError("the evidence has probability 0 under every hypothesis")
    return [j / evidence for j in joint]


def repeated_test_posterior_solution(prior, sensitivity, false_positive_rate, positives):
    if positives < 0:
        raise ValueError(f"the number of positive results cannot be negative, got {positives}")
    belief = prior
    for _ in range(positives):
        belief = bayes_posterior(belief, sensitivity, false_positive_rate)  # yesterday's posterior is today's prior
    return belief


check("posterior_solution", posterior_solution, POSTERIOR_CASES, strict=True)
check("repeated_test_posterior_solution", repeated_test_posterior_solution, REPEATED_TEST_CASES, strict=True)

In [ ]:
def exponential_log_likelihood_solution(rate, data):
    if rate <= 0:
        raise ValueError(f"the rate must be positive, got {rate}")
    if len(data) == 0:
        raise ValueError("need at least one waiting time")
    if any(x < 0 for x in data):
        raise ValueError("waiting times cannot be negative")
    return len(data) * math.log(rate) - rate * math.fsum(data)


def exponential_mle_solution(data):
    # d/d(rate) [n log(rate) - rate * sum(x)] = n / rate - sum(x) = 0, so rate = n / sum(x) = 1 / mean.
    if len(data) == 0:
        raise ValueError("cannot estimate a rate from no data")
    if any(x < 0 for x in data):
        raise ValueError("waiting times cannot be negative")
    total = math.fsum(data)
    if total == 0:
        raise ValueError("all waiting times are zero, so the estimated rate would be infinite")
    return len(data) / total


check("exponential_log_likelihood_solution", exponential_log_likelihood_solution, EXPONENTIAL_LL_CASES, strict=True)
check("exponential_mle_solution", exponential_mle_solution, EXPONENTIAL_MLE_CASES, strict=True)

# The estimate really does maximise the log-likelihood: nearby rates all score lower.
waits = [0.8, 2.1, 0.3, 1.7, 0.6, 3.2, 1.1]
rate_hat = exponential_mle_solution(waits)
peak = exponential_log_likelihood_solution(rate_hat, waits)
assert all(exponential_log_likelihood_solution(rate_hat * factor, waits) < peak for factor in (0.8, 0.9, 1.1, 1.2)), "the MLE should be a maximum"
print(f"rate_hat = {rate_hat:.4f} for a mean waiting time of {mean(waits):.4f}")

**Expected behaviour.** Every solution prints a pass line, such as `posterior_solution: all 7 tests passed`, and the final cell confirms that $\hat\lambda = 1/\bar{x}$ beats nearby rates. If you change a solution so that it is wrong (try dividing by `len(data) - 1` in `exponential_mle_solution`), its `check` call raises `AssertionError`, which shows that the tests can tell right answers from wrong ones.